# 06 — Evaluation: RF-DETR vs YOLOv8n

Deep-dive evaluation of the RF-DETR Nano model on the test split, with a head-to-head comparison against YOLOv8n.

**Model:** RF-DETR Nano (`checkpoint_best_ema.pth`) — best val/ema_mAP_50 checkpoint (epoch 11, 0.970)  
**Dataset:** 21 test images, class: `bib`  
**YOLOv8n baseline:** mAP50=0.940, Precision=0.960, Recall=0.957, F1=0.932

## Setup

In [ ]:
from pathlib import Path
import numpy as np
import torch
import supervision as sv
from PIL import Image
from rfdetr import RFDETRNano

DEVICE      = "mps" if torch.backends.mps.is_available() else "cpu"
DATASET_DIR = Path("../data/labeled/race-vision.v1i.yolov8").resolve()
RUNS_DIR    = Path("../models/runs/rfdetr-nano-map50").resolve()
ARTIFACTS   = Path("artifacts/rfdetr-nano-map50")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

CONF_THRESHOLD = 0.5   # same threshold used during val in notebook 05
IOU_THRESHOLD  = 0.5   # IoU required to count a detection as a true positive

best_weights = RUNS_DIR / "checkpoint_best_ema.pth"
assert best_weights.exists(), f"Weights not found: {best_weights}"

model = RFDETRNano(pretrain_weights=str(best_weights))
print(f"Loaded: {best_weights.name}")

TEST_IMG_DIR = DATASET_DIR / "test" / "images"
TEST_LBL_DIR = DATASET_DIR / "test" / "labels"
test_images  = sorted(TEST_IMG_DIR.glob("*.*"))
print(f"Test images: {len(test_images)}")

## Run inference on test split

In [ ]:
def load_yolo_labels(label_path, img_w, img_h):
    """Parse YOLO-format label file → xyxy boxes (absolute pixels)."""
    boxes = []
    if not label_path.exists():
        return np.zeros((0, 4))
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, cx, cy, bw, bh = map(float, parts[:5])
            x1 = (cx - bw / 2) * img_w
            y1 = (cy - bh / 2) * img_h
            x2 = (cx + bw / 2) * img_w
            y2 = (cy + bh / 2) * img_h
            boxes.append([x1, y1, x2, y2])
    return np.array(boxes, dtype=float) if boxes else np.zeros((0, 4))


def box_iou(boxA, boxB):
    """Compute IoU between two xyxy boxes."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0


records = []   # per-detection records for later analysis

total_gt    = 0
total_tp    = 0
total_fp    = 0
total_fn    = 0

all_predictions = []
all_targets     = []

for img_path in test_images:
    img = Image.open(img_path).convert("RGB")
    img_w, img_h = img.size

    detections = model.predict(img, threshold=CONF_THRESHOLD)
    pred_boxes = detections.xyxy if len(detections) else np.zeros((0, 4))
    pred_confs = detections.confidence if detections.confidence is not None else np.zeros(0)

    lbl_path = TEST_LBL_DIR / (img_path.stem + ".txt")
    gt_boxes = load_yolo_labels(lbl_path, img_w, img_h)

    total_gt += len(gt_boxes)

    matched_gt = set()
    for pi, (pb, pc) in enumerate(zip(pred_boxes, pred_confs)):
        best_iou, best_gi = 0.0, -1
        for gi, gb in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            iou = box_iou(pb, gb)
            if iou > best_iou:
                best_iou, best_gi = iou, gi

        is_tp = best_iou >= IOU_THRESHOLD and best_gi >= 0
        if is_tp:
            matched_gt.add(best_gi)
            total_tp += 1
        else:
            total_fp += 1

        gt_area = 0
        if best_gi >= 0 and len(gt_boxes):
            b = gt_boxes[best_gi]
            gt_area = (b[2] - b[0]) * (b[3] - b[1])

        records.append({
            "image": img_path.name,
            "conf": pc,
            "is_tp": is_tp,
            "best_iou": best_iou,
            "gt_area": gt_area,
            "pred_area": (pb[2] - pb[0]) * (pb[3] - pb[1]),
        })

    total_fn += len(gt_boxes) - len(matched_gt)

    gt_class_ids = np.zeros(len(gt_boxes), dtype=int) if len(gt_boxes) else np.zeros(0, dtype=int)
    all_targets.append(sv.Detections(xyxy=gt_boxes, class_id=gt_class_ids))
    all_predictions.append(detections)

precision  = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
recall     = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
f1         = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"GT boxes:    {total_gt}")
print(f"Predictions: {total_tp + total_fp}")
print(f"TP: {total_tp}  FP: {total_fp}  FN: {total_fn}")
print(f"Precision:   {precision:.3f}")
print(f"Recall:      {recall:.3f}")
print(f"F1:          {f1:.3f}")

## mAP via supervision

In [ ]:
# mAP requires the full precision-recall curve — pass all detections at a low threshold
# so supervision can sweep confidence internally. Using CONF_THRESHOLD=0.5 here would
# truncate the recall axis and give a falsely low mAP.
MAP_THRESHOLD = 0.01

map_metric = sv.metrics.MeanAveragePrecision()

for img_path in test_images:
    img = Image.open(img_path).convert("RGB")
    img_w, img_h = img.size
    preds_low = model.predict(img, threshold=MAP_THRESHOLD)
    lbl_path = TEST_LBL_DIR / (img_path.stem + ".txt")
    gt_boxes = load_yolo_labels(lbl_path, img_w, img_h)
    targets = sv.Detections(
        xyxy=gt_boxes,
        class_id=np.zeros(len(gt_boxes), dtype=int) if len(gt_boxes) else np.zeros(0, dtype=int),
    )
    map_metric.update([preds_low], [targets])

result = map_metric.compute()
map50    = result.map50
map50_95 = result.map50_95

print(f"mAP50:    {map50:.3f}")
print(f"mAP50-95: {map50_95:.3f}")

## Confidence distribution: TP vs FP

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(records)

tp_conf = df[df["is_tp"]]["conf"]
fp_conf = df[~df["is_tp"]]["conf"]

print(f"TP — count: {len(tp_conf)}, avg conf: {tp_conf.mean():.3f}, min: {tp_conf.min():.3f}")
print(f"FP — count: {len(fp_conf)}, avg conf: {fp_conf.mean():.3f}, min: {fp_conf.min():.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 1, 21)
ax.hist(tp_conf, bins=bins, alpha=0.7, label=f"TP (n={len(tp_conf)})")
ax.hist(fp_conf, bins=bins, alpha=0.7, label=f"FP (n={len(fp_conf)})")
ax.axvline(CONF_THRESHOLD, color="red", linestyle="--", label=f"threshold={CONF_THRESHOLD}")
ax.set_xlabel("Confidence")
ax.set_ylabel("Count")
ax.set_title("RF-DETR: Confidence Distribution — TP vs FP")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
out = ARTIFACTS / "confidence_distribution.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")

## Box-size analysis: what did the model miss?

Missed detections (FNs) are bibs in the label set that no prediction matched. We can approximate their size from the label file.

In [ ]:
# Collect sizes for all GT boxes and for unmatched GT boxes (FNs)
gt_areas_all = []
fn_areas     = []

for img_path in test_images:
    img = Image.open(img_path).convert("RGB")
    img_w, img_h = img.size

    detections = model.predict(img, threshold=CONF_THRESHOLD)
    pred_boxes = detections.xyxy if len(detections) else np.zeros((0, 4))
    pred_confs = detections.confidence if detections.confidence is not None else np.zeros(0)

    lbl_path = TEST_LBL_DIR / (img_path.stem + ".txt")
    gt_boxes = load_yolo_labels(lbl_path, img_w, img_h)

    matched_gt = set()
    for pb, pc in zip(pred_boxes, pred_confs):
        best_iou, best_gi = 0.0, -1
        for gi, gb in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            iou = box_iou(pb, gb)
            if iou > best_iou:
                best_iou, best_gi = iou, gi
        if best_iou >= IOU_THRESHOLD and best_gi >= 0:
            matched_gt.add(best_gi)

    for gi, gb in enumerate(gt_boxes):
        area = (gb[2] - gb[0]) * (gb[3] - gb[1])
        gt_areas_all.append(area)
        if gi not in matched_gt:
            fn_areas.append(area)

print(f"All GT areas — median: {np.median(gt_areas_all):.0f}px², mean: {np.mean(gt_areas_all):.0f}px²")
if fn_areas:
    print(f"FN areas     — median: {np.median(fn_areas):.0f}px², mean: {np.mean(fn_areas):.0f}px²")
else:
    print("No false negatives — all bibs detected.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(gt_areas_all, bins=20, alpha=0.6, label="All GT bibs")
if fn_areas:
    ax.hist(fn_areas, bins=10, alpha=0.8, label=f"Missed bibs (FN, n={len(fn_areas)})")
ax.set_xlabel("Bounding box area (px²)")
ax.set_ylabel("Count")
ax.set_title("RF-DETR: Bib Size Distribution vs Missed Detections")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
out = ARTIFACTS / "box_size_analysis.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")

## False positives: spot-check

In [ ]:
import math

fp_images = df[~df["is_tp"]]["image"].unique().tolist()
print(f"Images with FPs: {len(fp_images)}")

if fp_images:
    annotator = sv.BoxAnnotator(color=sv.Color.RED)
    gt_annotator = sv.BoxAnnotator(color=sv.Color.GREEN)

    COLS = min(3, len(fp_images))
    rows = math.ceil(len(fp_images) / COLS)
    fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 5, rows * 4))
    axes = np.array(axes).flatten() if rows * COLS > 1 else [axes]

    for ax, img_name in zip(axes, fp_images):
        img_path = TEST_IMG_DIR / img_name
        img = Image.open(img_path).convert("RGB")
        img_w, img_h = img.size

        detections = model.predict(img, threshold=CONF_THRESHOLD)
        pred_boxes = detections.xyxy if len(detections) else np.zeros((0, 4))
        pred_confs = detections.confidence if detections.confidence is not None else np.zeros(0)

        lbl_path = TEST_LBL_DIR / (img_path.stem + ".txt")
        gt_boxes = load_yolo_labels(lbl_path, img_w, img_h)

        img_np = np.array(img)
        if len(gt_boxes):
            gt_det = sv.Detections(xyxy=gt_boxes, class_id=np.zeros(len(gt_boxes), dtype=int))
            img_np = gt_annotator.annotate(scene=img_np, detections=gt_det)
        if len(detections):
            img_np = annotator.annotate(scene=img_np, detections=detections)

        ax.imshow(img_np)
        ax.set_title(f"{img_name[:25]}\nred=pred, green=GT", fontsize=8)
        ax.axis("off")

    for ax in axes[len(fp_images):]:
        ax.axis("off")

    plt.tight_layout()
    out = ARTIFACTS / "false_positives.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")
else:
    print("No false positives to display.")

## False negatives: missed detections

In [ ]:
# Re-collect FN image list
fn_image_list = []

for img_path in test_images:
    img = Image.open(img_path).convert("RGB")
    img_w, img_h = img.size

    detections = model.predict(img, threshold=CONF_THRESHOLD)
    pred_boxes = detections.xyxy if len(detections) else np.zeros((0, 4))

    lbl_path = TEST_LBL_DIR / (img_path.stem + ".txt")
    gt_boxes = load_yolo_labels(lbl_path, img_w, img_h)

    matched_gt = set()
    for pb in pred_boxes:
        best_iou, best_gi = 0.0, -1
        for gi, gb in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            iou = box_iou(pb, gb)
            if iou > best_iou:
                best_iou, best_gi = iou, gi
        if best_iou >= IOU_THRESHOLD and best_gi >= 0:
            matched_gt.add(best_gi)

    if len(gt_boxes) - len(matched_gt) > 0:
        fn_image_list.append((img_path, gt_boxes, list(set(range(len(gt_boxes))) - matched_gt)))

print(f"Images with FNs: {len(fn_image_list)}")

if fn_image_list:
    COLS = min(3, len(fn_image_list))
    rows = math.ceil(len(fn_image_list) / COLS)
    fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 5, rows * 4))
    axes = np.array(axes).flatten() if rows * COLS > 1 else [axes]

    gt_annotator = sv.BoxAnnotator(color=sv.Color.GREEN)
    miss_annotator = sv.BoxAnnotator(color=sv.Color.RED)

    for ax, (img_path, gt_boxes, fn_indices) in zip(axes, fn_image_list):
        img = Image.open(img_path).convert("RGB")
        img_np = np.array(img)

        # Draw all GT in green
        gt_det = sv.Detections(xyxy=gt_boxes, class_id=np.zeros(len(gt_boxes), dtype=int))
        img_np = gt_annotator.annotate(scene=img_np, detections=gt_det)

        # Overlay missed GT in red
        missed_boxes = gt_boxes[fn_indices]
        miss_det = sv.Detections(xyxy=missed_boxes, class_id=np.zeros(len(missed_boxes), dtype=int))
        img_np = miss_annotator.annotate(scene=img_np, detections=miss_det)

        ax.imshow(img_np)
        ax.set_title(f"{img_path.name[:25]}\ngreen=detected, red=missed", fontsize=8)
        ax.axis("off")

    for ax in axes[len(fn_image_list):]:
        ax.axis("off")

    plt.tight_layout()
    out = ARTIFACTS / "false_negatives.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")
else:
    print("No false negatives — all bibs detected!")

## Head-to-head: RF-DETR Nano vs YOLOv8n

YOLOv8n baseline numbers from `04_evaluation.ipynb` (test split, conf ≥ 0.25, IoU ≥ 0.5).

In [ ]:
import pandas as pd

# YOLOv8n test-split results (from 04_evaluation.ipynb)
yolo_results = {
    "Model":       "YOLOv8n",
    "Params":      "3.0M",
    "mAP50":       0.940,
    "mAP50-95":    0.619,
    "Precision":   0.960,
    "Recall":      0.957,
    "F1":          0.932,
    "TP":          48,
    "FP":          5,
    "FN":          2,
    "License":     "AGPL-3.0",
}

rfdetr_results = {
    "Model":       "RF-DETR Nano",
    "Params":      "6.5M",
    "mAP50":       round(map50, 3),
    "mAP50-95":    round(map50_95, 3),
    "Precision":   round(precision, 3),
    "Recall":      round(recall, 3),
    "F1":          round(f1, 3),
    "TP":          total_tp,
    "FP":          total_fp,
    "FN":          total_fn,
    "License":     "Apache 2.0",
}

comparison = pd.DataFrame([yolo_results, rfdetr_results]).set_index("Model")
print(comparison.to_string())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics    = ["mAP50", "mAP50-95", "Precision", "Recall", "F1"]
yolo_vals  = [yolo_results[m] for m in metrics]
rfdetr_vals = [rfdetr_results[m] for m in metrics]

x   = np.arange(len(metrics))
w   = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - w/2, yolo_vals,   w, label="YOLOv8n (AGPL-3.0)")
bars2 = ax.bar(x + w/2, rfdetr_vals, w, label="RF-DETR Nano (Apache 2.0)")

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("RF-DETR Nano vs YOLOv8n — Test Split")
ax.legend()
ax.grid(axis="y", alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
out = ARTIFACTS / "model_comparison.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")

## Summary

| Artifact | Path |
|---|---|
| Confidence distribution | `notebooks/artifacts/rfdetr-nano/confidence_distribution.png` |
| Box size analysis | `notebooks/artifacts/rfdetr-nano/box_size_analysis.png` |
| False positives | `notebooks/artifacts/rfdetr-nano/false_positives.png` |
| False negatives | `notebooks/artifacts/rfdetr-nano/false_negatives.png` |
| Model comparison chart | `notebooks/artifacts/rfdetr-nano/model_comparison.png` |